In [194]:
# data analysis
import pandas as pd
import numpy as np
import random

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pyampute.exploration.md_patterns import mdPatterns
from pyampute.exploration.mcar_statistical_tests import MCARTest
import missingno as msno


# preprocessing
import sklearn.utils.validation
import sys
from scipy import stats
from scipy.stats import shapiro, distributions, loguniform
from scipy.stats.mstats import winsorize
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, HalvingRandomSearchCV, HalvingGridSearchCV, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, QuantileTransformer, MinMaxScaler, KBinsDiscretizer, Binarizer, PolynomialFeatures, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from feature_engine.outliers import Winsorizer
# from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.pipeline import Pipeline
from sklearn import set_config

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# Modeling
from sklearn.linear_model import RidgeClassifier, LogisticRegression, RidgeClassifierCV, LogisticRegressionCV, SGDClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
import joblib
from sklearn import tree 
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier 

# Metrics
from sklearn.metrics import confusion_matrix, recall_score, precision_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report, precision_recall_curve, PrecisionRecallDisplay, log_loss, brier_score_loss, roc_curve, roc_auc_score, RocCurveDisplay, det_curve, DetCurveDisplay, fbeta_score, average_precision_score, matthews_corrcoef

# Calibration
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV

# Inspection
from sklearn.inspection import PartialDependenceDisplay

# Custom Functions
from credit_risk_modeling import model_eval, scoring, decision_rules
import importlib
# importlib.reload(model_eval)
importlib.reload(scoring)

<module 'credit_risk_modeling.scoring' from 'C:\\Users\\billy\\OneDrive\\Documents\\Finance_Projects\\credit_risk_modeling\\src\\credit_risk_modeling\\scoring.py'>

### Score Applicant

In [195]:
interim_df = pd.read_csv(
    "../data/interim/credit_risk_dataset_prepped.csv"
)

In [196]:
applicant_idx = 0

In [197]:
applicant = interim_df.iloc[applicant_idx].drop("loan_status")

In [198]:
applicant

person_age                           21
person_income                      9600
person_home_ownership               OWN
person_emp_length                   5.0
loan_intent                   EDUCATION
loan_grade                            B
loan_amnt                          1000
loan_int_rate                     11.14
loan_percent_income                 0.1
cb_person_default_on_file         False
cb_person_cred_hist_length            2
Name: 0, dtype: object

In [199]:
score = scoring.score_applicant(features=applicant, phase=3)


2026-02-10 17:00:00.250 | INFO     | credit_risk_modeling.scoring:load_model_and_preprocessor:15 - ✓ Model and preprocessor loaded successfully
2026-02-10 17:00:00.286 | WARNING  | credit_risk_modeling.scoring:get_lgd_by_segment:89 - LGD file not found, defaulting to 1.0


c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [200]:
score_df = pd.DataFrame([score])
score_df

,pd,lgd,ead,expected_loss,risk_score,risk_tier,confidence
0,0.04052,1.0,1000.0,40.520142,4,LOW,0.08104


In [201]:
score_df.pd.values[0]

0.040520142391324045

In [202]:
score_df['pd']

0    0.04052
Name: pd, dtype: float64

### Approval

In [203]:
engine = decision_rules.ApprovalRuleEngine()

2026-02-10 17:00:00.340 | INFO     | credit_risk_modeling.decision_rules:__init__:46 - ✓ ApprovalRuleEngine initialized with thresholds: auto_approve=5%, manual_review=15%


In [204]:
decision = engine.decide(
    pd = score_df.pd.values[0],
    lgd = score_df.lgd.values[0],
    ead = score_df.ead.values[0],
    expected_loss = score_df.expected_loss.values[0]
)

2026-02-10 17:00:00.350 | INFO     | credit_risk_modeling.decision_rules:decide:98 - Decision: APPROVE | Applicant: UNKNOWN | EL: 4.05% | Reason: Auto-approved: Expected Loss of 4.05% is below auto-approve threshold (5.00%) fo...


### Batch approval

In [205]:
batch_indice = [random.randint(1, 100) for _ in range(20)]

In [206]:
batch = [interim_df.iloc[applicant].drop("loan_status") for applicant in batch_indice]

In [207]:
batch_score = scoring.score_batch(
    applicants_list= batch
)

2026-02-10 17:00:00.425 | INFO     | credit_risk_modeling.scoring:load_model_and_preprocessor:15 - ✓ Model and preprocessor loaded successfully


c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not

In [208]:
batch_score

[{'pd': 0.014822314865887165,
  'lgd': 1.0,
  'ead': 28000.0,
  'expected_loss': 415.0248162448406,
  'risk_score': 1,
  'risk_tier': 'LOW',
  'confidence': 0.029644629731774286},
 {'pd': 1.0,
  'lgd': 1.0,
  'ead': 25475.0,
  'expected_loss': 25475.0,
  'risk_score': 100,
  'risk_tier': 'HIGH',
  'confidence': 0.0},
 {'pd': 0.9377076983451843,
  'lgd': 1.0,
  'ead': 1600.0,
  'expected_loss': 1500.332317352295,
  'risk_score': 93,
  'risk_tier': 'HIGH',
  'confidence': 0.12458460330963139},
 {'pd': 0.0010810811072587967,
  'lgd': 1.0,
  'ead': 20000.0,
  'expected_loss': 21.621622145175934,
  'risk_score': 0,
  'risk_tier': 'LOW',
  'confidence': 0.0021621622145175934},
 {'pd': 0.06445658877491951,
  'lgd': 1.0,
  'ead': 20000.0,
  'expected_loss': 1289.1317754983902,
  'risk_score': 6,
  'risk_tier': 'LOW',
  'confidence': 0.128913177549839},
 {'pd': 0.0,
  'lgd': 1.0,
  'ead': 35000.0,
  'expected_loss': 0.0,
  'risk_score': 0,
  'risk_tier': 'LOW',
  'confidence': 0.0},
 {'pd': 0.0

In [209]:
for applicant in batch_score:
    value = applicant['pd']
    print(value)

0.014822314865887165
1.0
0.9377076983451843
0.0010810811072587967
0.06445658877491951
0.0
0.001433691754937172
1.0
0.012290155421942473
1.0
1.0
1.0
0.020025225169956685
1.0
1.0
0.9882352948188782
0.025034718774259092
1.0
0.01869851052761078
0.9746920466423035


In [210]:
engine.batch_decide(
    scoring_results= batch_score
)

2026-02-10 17:05:18.693 | INFO     | credit_risk_modeling.decision_rules:decide:98 - Decision: APPROVE | Applicant: UNKNOWN | EL: 1.48% | Reason: Auto-approved: Expected Loss of 1.48% is below auto-approve threshold (5.00%) fo...
2026-02-10 17:05:18.693 | INFO     | credit_risk_modeling.decision_rules:decide:98 - Decision: DENY | Applicant: UNKNOWN | EL: 100.00% | Reason: Auto-denied: Expected Loss of 100.00% exceeds manual-review threshold (15.00%) f...
2026-02-10 17:05:18.693 | INFO     | credit_risk_modeling.decision_rules:decide:98 - Decision: DENY | Applicant: UNKNOWN | EL: 93.77% | Reason: Auto-denied: Expected Loss of 93.77% exceeds manual-review threshold (15.00%) fo...
2026-02-10 17:05:18.693 | INFO     | credit_risk_modeling.decision_rules:decide:98 - Decision: APPROVE | Applicant: UNKNOWN | EL: 0.11% | Reason: Auto-approved: Expected Loss of 0.11% is below auto-approve threshold (5.00%) fo...
2026-02-10 17:05:18.693 | INFO     | credit_risk_modeling.decision_rules:decide:98 

[{'pd': 0.014822314865887165,
  'lgd': 1.0,
  'ead': 28000.0,
  'expected_loss': 415.0248162448406,
  'risk_score': 1,
  'risk_tier': 'LOW',
  'confidence': 0.029644629731774286,
  'decision': 'APPROVE',
  'reason': 'Auto-approved: Expected Loss of 1.48% is below auto-approve threshold (5.00%) for UNKNOWN loans. PD=1.48%, LGD=100%, EAD=$28,000',
  'el_pct': 0.014822314865887165},
 {'pd': 1.0,
  'lgd': 1.0,
  'ead': 25475.0,
  'expected_loss': 25475.0,
  'risk_score': 100,
  'risk_tier': 'HIGH',
  'confidence': 0.0,
  'decision': 'DENY',
  'reason': 'Auto-denied: Expected Loss of 100.00% exceeds manual-review threshold (15.00%) for UNKNOWN loans. Risk unacceptable. PD=100.00%, LGD=100%, EAD=$25,475',
  'el_pct': 1.0},
 {'pd': 0.9377076983451843,
  'lgd': 1.0,
  'ead': 1600.0,
  'expected_loss': 1500.332317352295,
  'risk_score': 93,
  'risk_tier': 'HIGH',
  'confidence': 0.12458460330963139,
  'decision': 'DENY',
  'reason': 'Auto-denied: Expected Loss of 93.77% exceeds manual-review th